# EZ-VC Kaggle Inference

This notebook sets up `ez-vc/ez-vc` on Kaggle, installs the dependencies needed for the XEUS inference path, fetches the gated Hugging Face assets, and writes a converted WAV.

Before running: add a Kaggle secret named `HF_TOKEN` with access to `SPRINGLab/EZ-VC`.

In [ ]:
from pathlib import Path
import os

WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'ez-vc'
OUT_DIR = WORKDIR / 'ezvc_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token loaded from Kaggle secret HF_TOKEN')
except Exception as exc:
    print('HF_TOKEN Kaggle secret not loaded:', exc)
    print('Set os.environ["HF_TOKEN"] manually before downloading gated assets.')


In [ ]:
%cd /kaggle/working
![ -d ez-vc/.git ] || git clone https://github.com/RahulBhalley/EZ-VC.git ez-vc
%cd /kaggle/working/ez-vc
!git pull --ff-only


## Install dependencies

The README-pinned ESPnet fork declares old NumPy constraints that do not work cleanly on modern Kaggle Python images. The workaround below installs the exact ESPnet code without its incompatible dependency pins, then installs the runtime packages needed by the XEUS path.

In [ ]:
%cd /kaggle/working/ez-vc
%pip install -q --upgrade pip setuptools wheel
%pip install -q -e .
%pip install -q torchcodec
%pip install -q --no-deps 'espnet @ git+https://github.com/wanchichen/espnet.git@ssl'
%pip install -q configargparse typeguard humanfriendly librosa==0.9.2 jamo h5py kaldiio torch_complex nltk
%pip install -q g2p_en espnet_tts_frontend opt-einsum editdistance sentencepiece


## Patch vendored BigVGAN for current Hugging Face Hub

Recent `huggingface_hub` versions no longer pass a few keyword arguments that vendored BigVGAN marks as required. The patch only adds safe defaults.

In [ ]:
from pathlib import Path

bigvgan_py = Path('/kaggle/working/ez-vc/src/third_party/BigVGAN/bigvgan.py')
text = bigvgan_py.read_text()
text = text.replace('proxies: Optional[Dict],', 'proxies: Optional[Dict] = None,')
text = text.replace('resume_download: bool,', 'resume_download: Optional[bool] = None,')
text = text.replace('local_files_only: bool,', 'local_files_only: bool = False,')
text = text.replace('token: Union[str, bool, None],', 'token: Union[str, bool, None] = None,')
bigvgan_py.write_text(text)
print('Patched', bigvgan_py)


In [ ]:
import torch
import torchaudio
import huggingface_hub

print('torch', torch.__version__)
print('torchaudio', torchaudio.__version__)
print('huggingface_hub', huggingface_hub.__version__)
print('cuda available', torch.cuda.is_available())


## Run EZ-VC inference

This uses the bundled target-speaker and source-speech examples. Replace `ref_audio` and `src_wav` with uploaded Kaggle file paths for your own conversion.

In [ ]:
%cd /kaggle/working/ez-vc/src/f5_tts/infer

import re
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from cached_path import cached_path
from hydra.utils import get_class
from omegaconf import OmegaConf

from f5_tts.infer.utils_infer import (
    cfg_strength,
    cross_fade_duration,
    fix_duration,
    infer_process,
    load_model,
    load_vocoder,
    speed,
    sway_sampling_coef,
    target_rms,
)
from f5_tts.infer.utils_xeus import ApplyKmeans, extract_units, load_xeus_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

vocoder_name = 'bigvgan'
config_file = '../configs/F5TTS_Base_EZ-VC.yaml'
ckpt_file = str(cached_path('hf://SPRINGLab/EZ-VC/model_2700000.safetensors'))
vocab_file = str(cached_path('hf://SPRINGLab/EZ-VC/vocab.txt'))

xeus_model = load_xeus_model(device).eval()
apply_kmeans = ApplyKmeans(device)
vocoder = load_vocoder(vocoder_name=vocoder_name, device=device)

model_cfg = OmegaConf.load(config_file)
model_cls = get_class(f'f5_tts.model.{model_cfg.model.backbone}')
ema_model = load_model(
    model_cls,
    model_cfg.model.arch,
    ckpt_file,
    mel_spec_type=vocoder_name,
    vocab_file=vocab_file,
    device=device,
)

ref_audio = 'examples/wavs/14_208_000042_000000.wav'
src_wav = 'examples/wavs/237_134493_000015_000004.wav'
ref_text = extract_units(ref_audio, xeus_model, apply_kmeans, device)
src_text = extract_units(src_wav, xeus_model, apply_kmeans, device)
print(f'ref_units={len(ref_text)} src_units={len(src_text)}')

generated_audio_segments = []
for text in re.split(r'(?=\[\w+\])', src_text):
    gen_text = re.sub(r'\[(\w+)\]', '', text).strip()
    if not gen_text:
        continue
    audio_segment, sample_rate, _ = infer_process(
        ref_audio,
        ref_text,
        gen_text,
        ema_model,
        vocoder,
        mel_spec_type=vocoder_name,
        target_rms=target_rms,
        cross_fade_duration=cross_fade_duration,
        nfe_step=12,
        cfg_strength=cfg_strength,
        sway_sampling_coef=sway_sampling_coef,
        speed=speed,
        fix_duration=fix_duration,
        device=device,
    )
    generated_audio_segments.append(audio_segment)

if not generated_audio_segments:
    raise RuntimeError('No audio generated')

gen_wav = np.concatenate(generated_audio_segments)
out_path = Path('/kaggle/working/ezvc_outputs/ezvc_sample.wav')
sf.write(out_path, gen_wav, sample_rate)
print('wrote', out_path)
print(f'samples={len(gen_wav)} sr={sample_rate} duration_sec={len(gen_wav) / sample_rate:.3f}')


In [ ]:
from IPython.display import Audio, display

display(Audio('/kaggle/working/ezvc_outputs/ezvc_sample.wav'))


## Free memory after generation

Run this cell when you are done generating. It aggressively releases model references and GPU cache inside the notebook process.

In [ ]:
import gc

for name in ['ema_model', 'vocoder', 'xeus_model', 'apply_kmeans']:
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('model references released')
